In [1]:
import plotly.graph_objects as go
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.decomposition import PCA
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE
import colorcet as cc

In [ ]:
%cd ..
from Models.GRU_encoder import GRUEncoder

c:\Users\Kayned\Documents\GitHub\LSTM-FAISS-DTW


In [ ]:
from TrainModel.train import training
training(GRUEncoder)

Train: 733 videos | Val: 335 videos | Classes: 32
------------------------------------------------------------------------------------------
Model: GRUEncoder      input= 144      hidden= 256   layers= 2   dropout= 0.4    device= cuda
Training: epochs= 500    lr= 0.0005    MultiSimilarityLoss: alpha= 2.0 beta= 40.0 base= 0.5
Batch Train: P = 16 classes x K = 4 = 64 samples
Batch Val:   P = 16 classes x K = 4 = 64 samples
------------------------------------------------------------------------------------------
Epoch   1 | train: 0.9872 | val: 0.9792 | lr: 4.99e-04 | val_acc: 32.24%  <- best
Epoch   2 | train: 0.9439 | val: 0.9732 | lr: 4.95e-04 | val_acc: 34.33%  <- best
Epoch   3 | train: 0.9294 | val: 0.9490 | lr: 4.88e-04 | val_acc: 34.03%
Epoch   4 | train: 0.9221 | val: 0.9474 | lr: 4.78e-04 | val_acc: 35.22%  <- best
Epoch   5 | train: 0.9147 | val: 0.9385 | lr: 4.67e-04 | val_acc: 34.03%
Epoch   6 | train: 0.9040 | val: 0.9562 | lr: 4.52e-04 | val_acc: 39.10%  <- best
Epoch   7 

In [4]:
model_name = GRUEncoder.__name__

logs = np.load(f"Models/{model_name}_Checkpoints/training_logs.npz", allow_pickle=True)
train = np.load(f"Models/{model_name}_Checkpoints/train_embeddings.npz", allow_pickle=True)
val   = np.load(f"Models/{model_name}_Checkpoints/val_embeddings.npz", allow_pickle=True)

In [5]:
best_epoch = int(logs["best_epoch"])
best_val_loss = float(logs["best_val_loss"])
best_val_accuracy = float(logs["best_val_accuracy"])
best_train_dist = logs["best_train_dists"]
best_val_dist = logs["best_val_dists"]

print(f"Best epoch: {best_epoch}")
print(f"Best val loss: {best_val_loss:.4f}")
print(f"Best val accuracy: {best_val_accuracy:.2%}")
print()
print(f"Best train distance:")
print(f"Positive: {best_train_dist[0]}")
print(f"Negative: {best_train_dist[1]}")
print(f"Difference: {best_train_dist[2]}")
print()
print(f"Best validation distance:")
print(f"Positive: {best_val_dist[0]}")
print(f"Negative: {best_val_dist[1]}")
print(f"Difference: {best_val_dist[2]}")


Best epoch: 165
Best val loss: 0.6433
Best val accuracy: 85.97%

Best train distance:
Positive: 0.34447944164276123
Negative: 1.4146144390106201
Difference: 1.0701349973678589

Best validation distance:
Positive: 0.6244425177574158
Negative: 1.4270436763763428
Difference: 0.8026012182235718


In [6]:
epochs = logs["epochs"]
train_loss = logs["train_loss"]
val_loss = logs["val_loss"]
val_acc = logs["val_acc"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=train_loss, mode="lines", name="Train Loss"))
fig.add_trace(go.Scatter(x=epochs, y=val_loss, mode="lines", name="Val Loss"))
fig.add_trace(go.Scatter(x=epochs, y=val_acc, mode="lines", name="Val Accuracy"))

fig.update_layout(
    title="LSTM Training Metrics over Epochs",
    xaxis_title="Epochs",
    yaxis_title="Values",
    hovermode="x unified"
)

fig.show()

In [7]:
embeddings = train["last_embeddings"]
labels = train["labels"]
id_to_label = train["id_to_label"]

class_names = [id_to_label[label] for label in labels]

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=42
)

embeddings_2d = tsne.fit_transform(embeddings)

df = pd.DataFrame({
    "TSNE1": embeddings_2d[:, 0],
    "TSNE2": embeddings_2d[:, 1],
    "Class": class_names
})

palette_32 = cc.glasbey[:32]

fig = px.scatter(
    df,
    x="TSNE1",
    y="TSNE2",
    color="Class",
    title="t-SNE of Train Embeddings",
    width=1000,
    height=1000,
    color_discrete_sequence=palette_32
)

fig.show()

In [8]:
embeddings = val["last_embeddings"]
labels = val["labels"]
id_to_label = val["id_to_label"]

class_names = [id_to_label[label] for label in labels]

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=42
)

embeddings_2d = tsne.fit_transform(embeddings)

df = pd.DataFrame({
    "TSNE1": embeddings_2d[:, 0],
    "TSNE2": embeddings_2d[:, 1],
    "Class": class_names
})

palette_32 = cc.glasbey[:32]

fig = px.scatter(
    df,
    x="TSNE1",
    y="TSNE2",
    color="Class",
    title="t-SNE of Validation Embeddings",
    width=1000,
    height=1000,
    color_discrete_sequence=palette_32
)

fig.show()

In [9]:
y_true = logs["best_labels"]
y_pred = logs["best_prediction"]

labels_names = val["id_to_label"]
labels_ids = np.arange(len(labels_names))

cm = confusion_matrix(y_true, y_pred, labels=labels_ids)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig = go.Figure(data=go.Heatmap(
    z = cm_norm,
    zmin = 0,
    zmax = 1,
    x = labels_names,
    y = labels_names,
    colorscale="turbo",
    text=cm,
    texttemplate="%{text}",
    hovertemplate="True: %{y}<br>Pred: %{x}<extra></extra>"
))

fig.update_layout(
    title="Prediction in Best Epoch",
    xaxis_title="Predicted label",
    yaxis_title="True label",
    width=1200,
    height=1000,
    margin=dict(l=150, r=50, t=80, b=150)
)

fig.show()

In [10]:
loss_gap = val_loss - train_loss

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=loss_gap, mode="lines", name="Val Loss - Train Loss"))

fig.update_layout(
    title="Loss Gap over Epochs",
    xaxis_title="Epochs",
    yaxis_title="Loss gap",
    hovermode="x unified"
)

In [11]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    labels=labels_ids,
    target_names=labels_names,
    zero_division=0
))

              precision    recall  f1-score   support

  BASKETBALL       1.00      0.77      0.87        13
         BEE       0.77      0.83      0.80        12
   BREAKFAST       0.75      0.75      0.75        12
        CALL       0.89      0.89      0.89         9
         CAR       0.90      1.00      0.95         9
       CAT 3       0.89      0.89      0.89         9
   CHRISTMAS       0.79      0.92      0.85        12
        DEAF       0.67      0.83      0.74        12
         DOG       0.76      0.93      0.84        14
         EAT       0.86      1.00      0.92        12
     FIREMAN       1.00      0.89      0.94         9
        FISH       1.00      0.78      0.88         9
    FOOTBALL       1.00      1.00      1.00         9
       GREEN       0.89      0.80      0.84        10
        HEAD       0.75      1.00      0.86         9
    HOSPITAL       1.00      0.75      0.86        12
         KID       1.00      1.00      1.00         9
       LUNCH       0.50    

In [12]:
from sklearn.metrics import recall_score

recalls = recall_score(
    y_true,
    y_pred,
    labels=labels_ids,
    average=None,
    zero_division=0
)

order = np.argsort(recalls)

fig = go.Figure(go.Bar(
    x=recalls[order],
    y=labels_names[order],
    orientation="h"
))

fig.update_layout(
    title="Recall per Class",
    xaxis_title="Recall",
    yaxis_title="Class",
    width=1000,
    height=800,
)

fig.show()

In [13]:
train_dists_mean = logs["train_dists"]
val_dists_mean = logs["val_dists"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=train_dists_mean[:, 0], mode="lines", name="Train Positive Distance"))
fig.add_trace(go.Scatter(x=epochs, y=train_dists_mean[:, 1], mode="lines", name="Train Negative Distance"))
fig.add_trace(go.Scatter(x=epochs, y=val_dists_mean[:, 0], mode="lines", name="Val Positive Distance"))
fig.add_trace(go.Scatter(x=epochs, y=val_dists_mean[:, 1], mode="lines", name="Val Negative Distance"))

fig.update_layout(
    title="Embedding Distances over Epochs",
    xaxis_title="Epochs",
    yaxis_title="Distance",
    hovermode="x unified",
)

fig.show()

In [14]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=train_dists_mean[:, 2], mode="lines", name="Train Difference"))
fig.add_trace(go.Scatter(x=epochs, y=val_dists_mean[:, 2], mode="lines", name="Val Difference"))

fig.update_layout(
    title="Embedding Distances between Positive and Negative Samples over Epochs",
    xaxis_title="Epochs",
    yaxis_title="Distance",
    hovermode="x unified",
)

fig.show()

In [15]:
lr = logs["lr"]

fig = go.Figure()

fig.add_trace(go.Scatter(x=epochs, y=lr, mode="lines", name="Train Loss"))

fig.update_layout(
    title="Learning Rate over Epochs",
    xaxis_title="Epochs",
    yaxis_title="Learnig Rate",
    hovermode="x unified",
)

fig.show()

In [16]:
val.close()
train.close()
logs.close()